# Stage1 早期SOXX披露篮子原始来源预检

假说：2011年原始报告可能补充N-PORT之前的披露篮子。仅检验来源及归档，不读取SOXX结果、不拟合或回测。

In [1]:
from pathlib import Path
import hashlib, json
import fitz

root = Path.cwd().resolve()
while not (root / "SPEC.md").is_file():
    if root.parent == root: raise RuntimeError("workspace root missing")
    root = root.parent
card = root / "research/stage1_legacy_holdings_source_probe_20261022"
plan = json.loads((card / "source_preflight_plan.json").read_text())
archive = json.loads((card / "nas_source_archive_receipt.json").read_text())
sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert plan["status"] == "frozen_source_only_probe"
assert archive["remote_hash_verified"] is True
for entry in archive["entries"]:
    source = root / entry["path"]
    assert source.stat().st_size == entry["bytes"] and sha(source) == entry["sha256"]
pdf_path = root / plan["local_raw_path"]
assert pdf_path.stat().st_size <= plan["max_response_bytes"]
pdf = fitz.open(pdf_path)
assert len(pdf) == 426
first, continuation = pdf[360].get_text(), pdf[361].get_text()
for page in (first, continuation):
    assert "PHLX SOX SEMICONDUCTOR SECTOR INDEX FUND" in page
    assert "October 31, 2011" in page
assert "SanDisk Corp." in first and "Advanced Micro Devices Inc." in first
assert "NVIDIA Corp." in first and "Taiwan Semiconductor Manufacturing Co. Ltd." in first
assert "Shares" in first and "Value" in first and "TOTAL COMMON STOCKS" in first
assert "NET ASSETS" in continuation
headers = (root / "data/snapshots/stage1_legacy_holdings_source_probe_20261022/2011_10_31_headers.txt").read_text().lower()
assert "http/2 200" in headers and "content-type: application/pdf" in headers
assert "last-modified: tue, 21 feb 2023" in headers
result = {
    "status": "2011_SOXX_original_schedule_confirmed_full_Stage1_history_unqualified",
    "source_url": plan["source_url"],
    "source_pdf_sha256": sha(pdf_path),
    "source_pdf_bytes": pdf_path.stat().st_size,
    "http_headers_sha256": sha(root / "data/snapshots/stage1_legacy_holdings_source_probe_20261022/2011_10_31_headers.txt"),
    "page_count": len(pdf),
    "SOXX_schedule_pdf_pages_one_based": [361, 362],
    "report_as_of": "2011-10-31",
    "visible_columns": ["Security", "Shares", "Value"],
    "positive_control_names": ["SanDisk Corp.", "Advanced Micro Devices Inc.", "NVIDIA Corp.", "Taiwan Semiconductor Manufacturing Co. Ltd."],
    "PDF_creation_metadata": pdf.metadata.get("creationDate"),
    "ASX_url_calendar_date": "2012-01-03",
    "HTTP_last_modified": "2023-02-21 21:30:46 GMT",
    "earliest_public_time_qualified": False,
    "complete_2011_2022_report_inventory_qualified": False,
    "pre_2011_public_report_for_first_2011_Stage1_signal_qualified": False,
    "historical_constituent_price_and_identity_qualified": False,
    "Stage1_factor_values": 0,
    "market_outcome_reads": 0,
    "model_fits": 0,
    "native_accounts": 0,
    "SOXX_goal_achieved": False,
    "NAS_archive_sha256": archive["archive_sha256"],
    "NAS_remote_hash_verified": True,
    "next_action": "Inventory original 2010-2022 SOXX reports and filing clocks before the 2011-2022 Stage1 training signals; map every security identity and historical price source. Do not backfill a 2011-10-31 basket into earlier 2011 dates."
}
(card / "source_preflight_result.json").write_text(json.dumps(result, indent=2) + "\n")
summary = {
    "experiment": "stage1_legacy_holdings_source_probe_20261022",
    "decision": result["status"],
    "objective_achieved": False,
    "source_result": "source_preflight_result.json",
    "new_source_GETs": 1,
    "model_fits": 0,
    "native_accounts": 0,
    "artifact_policy": {"policy": "slim_card_recorder_first", "workflow_config": "workflow.yaml", "recorder_only_artifacts": []}
}
(card / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
conclusion = """# 原始2011年SOXX持仓表来源预检

按冻结计划只读取一份ASX托管的BlackRock/iShares原始PDF。HTTP 200、application/pdf，精确PDF为426页、3,412,380字节，SHA-256为`{pdf_sha}`；第361–362页明确标为2011-10-31的SOXX投资明细，含证券名称、股数和市值。SanDisk、AMD、NVIDIA和TSM ADR是可见正控。原PDF及响应头已归档NAS，远端SHA-256核对一致（`{archive_sha}`）。

PDF创建元数据为2012-01-03，ASX URL日期亦为2012-01-03；页面印刷流程时间在2011-12-22，现有HTTP Last-Modified为2023-02-21。三者均不能替代最早公开/接收时钟。现只证明**一份2011年原始报告存在**，尚未取得2010–2022全期报告、2011年首个信号以前的已公开篮子、所有成员的历史证券身份和价格。不得把10月底报告倒填到当年更早训练日。

决定：`2011_SOXX_original_schedule_confirmed_full_Stage1_history_unqualified`。0因子值、0市场收益读取、0拟合、0账户；SOXX稳定超额目标未完成。下一步先核对2010–2022原始报告及提交时钟和证券价格覆盖，再决定是否能冻结Stage1的匹配信息实验。
""".format(pdf_sha=result["source_pdf_sha256"], archive_sha=result["NAS_archive_sha256"])
(card / "conclusion.md").write_text(conclusion)
print(json.dumps({"status": result["status"], "pages": result["SOXX_schedule_pdf_pages_one_based"], "factor_values": 0}))

{"status": "2011_SOXX_original_schedule_confirmed_full_Stage1_history_unqualified", "pages": [361, 362], "factor_values": 0}


## 中文结论

执行后以 `conclusion.md`、`source_preflight_result.json` 与原始PDF/NAS归档回执为准；单份报告不构成完整Stage1训练历史或有效因子。